# Compare `Shoebox_6_patches` with ISM

This notebook generates RIR-like responses with MoD-ART/ART and with pyroomacoustics ISM using the same geometry, source-listener setup, and material coefficients from `materials.csv`. It then compares time responses and EDCs.

In [ ]:
import os
import time
import numpy as np
from pathlib import Path
import pyroomacoustics as pra
from typing import Tuple, List
from numpy.typing import NDArray, ArrayLike

import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline
from scipy.signal import butter, sosfilt
from scipy.io.wavfile import write

from raves import run_ART, run_MoDART
from slope2noise.utils import db, schroeder_backward_int, calculate_energy_envelope

# Compatibility patch for SciPy versions where mmread has no `spmatrix` keyword
import raves.src.runtime as _runtime
from scipy.io import mmread as _scipy_mmread

def _mmread_compat(path, spmatrix=True):
    return _scipy_mmread(path)

_runtime.mmread = _mmread_compat


In [ ]:
environment_folder = (Path('..') / 'example environments' / 'Shoebox_6_patches').resolve()
output_folder = Path('../../../Figures/ART/Comparison_with_ISM')
output_folder.mkdir(parents=True, exist_ok=True)


In [ ]:
# Shared simulation setup
source_position = np.array([1.25, 1.75, 1.50])
listener_position = np.array([3.25, 5.25, 1.50])

# Rates and duration
# - echogram_fs: ART/MoD-ART internal echogram rate
# - audio_fs: final rendered audio RIR rate
echogram_fs = 16000
audio_fs = 48000
rir_duration = 1.2
num_rays = 2000

# ISM order: chosen high enough to cover ~1.2 s in this shoebox
ism_max_order = 50

print(f'Source:   {source_position}')
print(f'Listener: {listener_position}')
print(f'echogram_fs={echogram_fs}, audio_fs={audio_fs}, rir_duration={rir_duration}s')


In [ ]:
# Read frequency-dependent absorption/scattering from the MoD-ART materials file
materials_file = environment_folder / 'materials.csv'
rows = [line.strip() for line in materials_file.read_text().splitlines() if line.strip()]

freqs = np.array([float(x.strip()) for x in rows[0].split(',')[1:]], dtype=float)

carpet_abs = np.array([float(x.strip()) for x in rows[1].split(',')[1:]], dtype=float)
carpet_scat = float(rows[2].split(',')[1].strip())
plaster_abs = np.array([float(x.strip()) for x in rows[3].split(',')[1:]], dtype=float)
plaster_scat = float(rows[4].split(',')[1].strip())
painted_concrete_abs = np.array([float(x.strip()) for x in rows[5].split(',')[1:]], dtype=float)
painted_concrete_scat = float(rows[6].split(',')[1].strip())

material_db = {
    'Carpet': {
        'energy_absorption': {
            'description': 'Carpet',
            'coeffs': carpet_abs.tolist(),
            'center_freqs': freqs.tolist(),
        },
        'scattering': {
            'description': 'Carpet',
            'coeffs': [carpet_scat] * len(freqs),
            'center_freqs': freqs.tolist(),
        },
    },
    'Plaster': {
        'energy_absorption': {
            'description': 'Plaster',
            'coeffs': plaster_abs.tolist(),
            'center_freqs': freqs.tolist(),
        },
        'scattering': {
            'description': 'Plaster',
            'coeffs': [plaster_scat] * len(freqs),
            'center_freqs': freqs.tolist(),
        },
    },
    'PaintedConcrete': {
        'energy_absorption': {
            'description': 'PaintedConcrete',
            'coeffs': painted_concrete_abs.tolist(),
            'center_freqs': freqs.tolist(),
        },
        'scattering': {
            'description': 'PaintedConcrete',
            'coeffs': [painted_concrete_scat] * len(freqs),
            'center_freqs': freqs.tolist(),
        },
    },
}

print('Frequencies:', freqs)
print('Carpet absorption:', carpet_abs)
print('Plaster absorption:', plaster_abs)
print('PaintedConcrete absorption:', painted_concrete_abs)


In [ ]:
# Run TD-ART and MoD-ART echograms for the same source-listener pair
source_positions = source_position[None, :]
listener_positions = listener_position[None, :]

start = time.time()
art_echograms, art_freqs = run_ART(
    str(environment_folder),
    source_positions,
    listener_positions,
    echogram_sample_rate=echogram_fs,
    echogram_duration=rir_duration,
    num_rays=num_rays,
    output_folder_path=str(output_folder),
)
art_runtime = time.time() - start

start = time.time()
modart_echograms, modart_freqs, modart_data = run_MoDART(
    str(environment_folder),
    source_positions,
    listener_positions,
    echogram_sample_rate=echogram_fs,
    echogram_duration=rir_duration,
    num_rays=num_rays,
    output_folder_path=str(output_folder),
)
modart_runtime = time.time() - start

assert np.allclose(art_freqs, modart_freqs)

print(f'ART runtime:    {art_runtime:.2f} s')
print(f'MoD-ART runtime:{modart_runtime:.2f} s')
print(f'Bands (Hz):     {art_freqs}')


In [ ]:
# Build ISM room with the same 4 x 7 x 3 shoebox and matching wall materials
# Mapping from mesh.obj:
# - floor: Carpet
# - ceiling: Plaster
# - west/north/east/south: PaintedConcrete
room_dim = np.array([4.0, 7.0, 3.0])

ism_materials = pra.make_materials(
    floor=(material_db['Carpet']['energy_absorption'], material_db['Carpet']['scattering']),
    ceiling=(material_db['Plaster']['energy_absorption'], material_db['Plaster']['scattering']),
    east=(material_db['PaintedConcrete']['energy_absorption'], material_db['PaintedConcrete']['scattering']),
    west=(material_db['PaintedConcrete']['energy_absorption'], material_db['PaintedConcrete']['scattering']),
    north=(material_db['PaintedConcrete']['energy_absorption'], material_db['PaintedConcrete']['scattering']),
    south=(material_db['PaintedConcrete']['energy_absorption'], material_db['PaintedConcrete']['scattering']),
)

room = pra.ShoeBox(
    room_dim,
    fs=audio_fs,
    materials=ism_materials,
    max_order=ism_max_order,
    air_absorption=True,
    ray_tracing=False,
    use_rand_ism = True,
)
# air absorption for temp = 20, humidity = 50
# room.set_air_absorption([0.3, 0.6, 1.0, 1.9, 5.8])

room.add_source(source_position)
room.add_microphone_array(np.c_[listener_position])
room.compute_rir()

ism_rir = np.asarray(room.rir[0][0], dtype=float)

# Match duration with ART/MoD-ART
n_samples_audio = int(np.round(rir_duration * audio_fs))
ism_rir = ism_rir[:n_samples_audio]
if len(ism_rir) < n_samples_audio:
    ism_rir = np.pad(ism_rir, (0, n_samples_audio - len(ism_rir)))

print('ISM RIR samples:', len(ism_rir))
print('ISM peak amplitude:', float(np.max(np.abs(ism_rir))))


In [ ]:
def bandpass_noise_shaped_rir(band_energy: NDArray, band_centers: List, fs, seed=0) -> Tuple[NDArray, ArrayLike]:
    """Render an audio-like RIR from per-band energy envelopes."""
    rng = np.random.default_rng(seed)
    n_bands, n_samples = band_energy.shape
    filtered_out = np.zeros_like(band_energy)
    t = np.arange(n_samples) / fs
    
    noise = rng.standard_normal(n_samples)
    noise = noise / np.sqrt(np.sum(noise**2) / len(noise))

    out = np.zeros(n_samples, dtype=float)
    band_bound = np.sqrt(2.0)

    for b in range(n_bands):
        low = band_centers[b] / band_bound
        high = band_centers[b] * band_bound
        if high >= 0.99 * (fs / 2):
            continue

        sos = butter(4, [low, high], btype='bandpass', output='sos', fs=fs)
        band_noise = sosfilt(sos, noise)
        env = np.sqrt(np.clip(band_energy[b], 0.0, None))
        filtered_out[b, :] = env * noise
        out += filtered_out[b, :]

    return filtered_out, out


def upsample_echogram(echogram, fs_in, fs_out, time_axis=-1):
    if fs_in == fs_out:
        return echogram
        # Take note of the echogram energy, to compare it after upsampling.
    old_energy = np.sum(echogram, axis=time_axis)

    # Prepare the audio-rate time intervals at which we'll evaluate the upsampled echogram.
    echogram_time_axis = np.arange(echogram.shape[-1]) / fs_in
    n_out = int(np.round(echogram.shape[time_axis] * fs_out / fs_in))
    rir_time_axis = np.arange(n_out) / fs_out
    # We use a linear interpolation, because any other upsampling algorithm risks introducing negative values.
    linear_spline = make_interp_spline(echogram_time_axis, echogram, k=1, axis=time_axis)
    upsampled_echogram = linear_spline(rir_time_axis)
    
    # Normalize w.r.t. the new sample rate, to preserve the energy-per-second definition of echogram values.
    upsampled_echogram *= fs_in / fs_out
    
    # Compare the new energy to the old one.
    new_energy = np.sum(upsampled_echogram, axis=-1)
    # The ratio (averaged over all frequency bands) should be close to 1 for all sources and listeners.
    print(f'Ratio of energies pre and post upsamling {np.mean(old_energy / new_energy, axis=time_axis)}')
    return upsampled_echogram


def rir_to_octave_energy(rir, fs, band_centers, smooth_win_ms:float=50) -> Tuple[NDArray, ArrayLike]:
    band_bound = np.sqrt(2.0)
    energies = []
    ir_len = len(rir)
    filtered_rirs = np.zeros((len(band_centers), ir_len), dtype=float)
    impulse_response = np.zeros_like(filtered_rirs)
    for b, fc in enumerate(band_centers):
        low = fc / band_bound
        high = fc * band_bound
        if high >= 0.99 * (fs / 2):
            energies.append(np.zeros_like(rir))
            continue
        sos = butter(4, [low, high], btype='bandpass', output='sos', fs=fs)
        filtered_rirs[b,:] = sosfilt(sos, rir)
        impulse_response[b, :] = sosfilt(sos, np.r_[1.0, np.zeros(ir_len - 1)])
        # compensate by filter energy
        filtered_rirs[b,:] /= np.sqrt(np.sum(impulse_response[b, :]**2))

    energies = calculate_energy_envelope(filtered_rirs**2, fs, smooth_time_ms=smooth_win_ms, time_axis=-1)
    return filtered_rirs, energies


In [ ]:
# Prepare comparable broadband energy responses
art_band_energy = np.clip(art_echograms[0, 0], 0.0, None)
modart_band_energy = np.clip(modart_echograms[0, 0], 0.0, None)

# Upsample band energies to audio_fs for rendering and fair visual comparison
art_band_energy_up = upsample_echogram(art_band_energy, echogram_fs, audio_fs)
modart_band_energy_up = upsample_echogram(modart_band_energy, echogram_fs, audio_fs)

# Convert ISM pressure RIR to band energies with matching bands
ism_subband_rirs, ism_band_energy = rir_to_octave_energy(ism_rir, audio_fs, art_freqs)

# Render audio-like RIRs from ART / MoD-ART band energies
art_subband_rirs, art_broadband_rir = bandpass_noise_shaped_rir(art_band_energy_up, art_freqs, audio_fs, seed=7452)
modart_subband_rirs, modart_broadband_rir = bandpass_noise_shaped_rir(modart_band_energy_up, art_freqs, audio_fs, seed=7452)

# Peak-normalized copies for WAV export
ism_rir_wav = ism_rir / (np.max(np.abs(ism_rir)) + 1e-12)
art_rir_wav = art_broadband_rir / (np.max(np.abs(art_broadband_rir)) + 1e-12)
modart_rir_wav = modart_broadband_rir / (np.max(np.abs(modart_broadband_rir)) + 1e-12)

write(output_folder / 'ism_rir.wav', audio_fs, ism_rir_wav.astype(np.float32))
write(output_folder / 'art_rir_noise_shaped.wav', audio_fs, art_rir_wav.astype(np.float32))
write(output_folder / 'modart_rir_noise_shaped.wav', audio_fs, modart_rir_wav.astype(np.float32))

np.save(output_folder / 'art_band_energy.npy', art_band_energy_up)
np.save(output_folder / 'modart_band_energy.npy', modart_band_energy_up)
np.save(output_folder / 'ism_band_energy.npy', ism_band_energy)

print('Saved:')
print('-', output_folder / 'ism_rir.wav')
print('-', output_folder / 'art_rir_noise_shaped.wav')
print('-', output_folder / 'modart_rir_noise_shaped.wav')


In [ ]:
# Compare broadband energy responses and EDCs
n = min(art_band_energy_up.shape[-1], modart_band_energy_up.shape[-1], ism_band_energy.shape[-1])
art_band_energy = art_band_energy_up[:,:n]
modart_band_energy = modart_band_energy_up[:,:n]
ism_band_energy = ism_band_energy[:,:n]
t = np.arange(n) / audio_fs
# Metrics in dB domain over first 1.0 s
valid = t <= min(1.0, rir_duration)

art_db = db(np.sum(art_band_energy, axis=0), is_squared=True)
modart_db = db(np.sum(modart_band_energy, axis=0), is_squared=True)
ism_db = db(np.sum(ism_band_energy,axis=0), is_squared=True)

art_band_energy_db = db(art_band_energy, is_squared=True)
modart_band_energy_db = db(modart_band_energy, is_squared=True)
ism_band_energy_db = db(ism_band_energy, is_squared=True)

# EDCs (for energy signals, pass sqrt(energy) so Schöder uses energy)
art_subband_edc = schroeder_backward_int(art_subband_rirs[:, :n])
modart_subband_edc = schroeder_backward_int(modart_subband_rirs[:, :n])
ism_subband_edc = schroeder_backward_int(ism_subband_rirs[:, :n])

art_edc_db = db(art_subband_edc, is_squared=True)
modart_edc_db = db(modart_subband_edc, is_squared=True)
ism_edc_db = db(ism_subband_edc, is_squared=True)

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def corr(a, b):
    a0 = a - np.mean(a)
    b0 = b - np.mean(b)
    denom = np.linalg.norm(a0) * np.linalg.norm(b0) + 1e-12
    return float(np.dot(a0, b0) / denom)

metrics = {
    'Broadband dB RMSE (ART vs ISM)': rmse(art_db, ism_db),
    'Broadband dB RMSE (MoD-ART vs ISM)': rmse(modart_db, ism_db),
    'Broadband dB corr (ART vs ISM)': corr(art_db, ism_db),
    'Broadband dB corr (MoD-ART vs ISM)': corr(modart_db, ism_db),
}

for k, v in metrics.items():
    print(f'{k}: {v:.3f}')


fig, ax = plt.subplots(len(freqs), 1, figsize=(11, 8), sharex=True)

for k in range(len(freqs)):
    ax[k].plot(t, ism_band_energy_db[k], label='ISM', lw=2)
    ax[k].plot(t, art_band_energy_db[k], label='ART', alpha=0.9)
    ax[k].plot(t, modart_band_energy_db[k], label='MoD-ART', alpha=0.9)
    ax[k].set_ylabel('Envelope [dB]')
    ax[k].set_xlabel('Time [s]')
    ax[k].set_title(f'Freq band = {freqs[k]:.0f} Hz')
    ax[k].set_ylim(max(art_band_energy_db[k]) - 80, max(art_band_energy_db[k])+3)
    ax[k].set_xlim([0, 1.2])
    ax[k].grid(True, alpha=0.3)
    ax[k].legend()
plt.tight_layout()
plt.savefig(output_folder / 'art_modart_vs_ism_echogram.png', dpi=180)


fig, ax = plt.subplots(len(freqs), 1, figsize=(11, 8), sharex=True)


for k in range(len(freqs)):
    ax[k].plot(t, ism_edc_db[k], label='ISM EDC', lw=2)
    ax[k].plot(t, art_edc_db[k], label='ART EDC', alpha=0.9)
    ax[k].plot(t, modart_edc_db[k], label='MoD-ART EDC', alpha=0.9)
    ax[k].set_ylabel('EDC [dB]')
    ax[k].set_xlabel('Time [s]')
    ax[k].set_title(f'Freq band = {freqs[k]:.0f} Hz')
    ax[k].set_ylim(max(art_edc_db[k]) - 80, max(art_edc_db[k])+3)
    ax[k].set_xlim([0, 1.2])
    ax[k].grid(True, alpha=0.3)
    ax[k].legend()

plt.tight_layout()
plt.savefig(output_folder / 'art_modart_vs_ism_edc.png', dpi=180)
plt.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
fig = go.Figure()

# Add traces
fig.add_trace(go.Scatter(x=t, y=ism_rir_wav, mode='lines', name='ISM'))
fig.add_trace(go.Scatter(x=t, y=art_rir_wav, mode='lines', name='ART'))
fig.add_trace(go.Scatter(x=t, y=modart_rir_wav, mode='lines', name='MoD-ART'))
fig.update_xaxes(range=[0, 1.0])

# Layout
fig.update_layout(
    title='Comparison of Room Impulse Responses',
    xaxis_title='Time [s]',
    yaxis_title='Amplitude',
    legend_title='RIR Type',
    template='plotly_white'
)